# SALT Industrial Inspector — MVTec AD Training

**Train SALT (Static-teacher Asymmetric Latent Training) on MVTec AD for industrial defect detection.**

This notebook:
1. Clones the SALT codebase from GitHub
2. Downloads MVTec AD dataset
3. Trains Stage 1 (MAE teacher) + Stage 2 (JEPA student) on all 15 categories
4. Exports ONNX model + reference embeddings for browser demo
5. Downloads all outputs

**Runtime**: Use GPU (T4 ≈ 2-3h, V100 ≈ 1h, A100 ≈ 30-45min)

Go to Runtime → Change runtime type → GPU

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone repository
!git clone https://github.com/MaxSikorski/salt-inspect.git
%cd salt-inspect

# Install dependencies
!pip install -q datasets Pillow

## 2. Download MVTec AD Dataset

In [ ]:
# Download MVTec AD (~5GB)
# Option A: From the official source (faster, direct download)
import os

DATA_ROOT = "/content/mvtec_data"
MVTEC_DIR = os.path.join(DATA_ROOT, "mvtec_anomaly_detection")

if not os.path.exists(MVTEC_DIR):
    os.makedirs(DATA_ROOT, exist_ok=True)
    
    # Download from official MVTec mirror
    !wget -q --show-progress -O /tmp/mvtec_ad.tar.xz \
        "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420938113-1629952094/mvtec_anomaly_detection.tar.xz"
    
    print("Extracting...")
    !cd {DATA_ROOT} && tar xf /tmp/mvtec_ad.tar.xz
    !rm /tmp/mvtec_ad.tar.xz
    print("Done!")
else:
    print(f"MVTec AD already exists at {MVTEC_DIR}")

# Verify
categories = sorted(os.listdir(MVTEC_DIR))
print(f"\nCategories ({len(categories)}): {categories}")

## 3. Configure Training

In [ ]:
# Training configuration
# Adjust these based on your GPU and time budget

STAGE1_EPOCHS = 200    # MAE teacher pretraining
STAGE2_EPOCHS = 200    # JEPA student training
BATCH_SIZE = 32        # Increase to 64 on A100
OUTPUT_DIR = "/content/salt-inspect/site"  # Output for web assets

# For quick test (5-10 min):
# STAGE1_EPOCHS = 20
# STAGE2_EPOCHS = 20

print(f"Stage 1: {STAGE1_EPOCHS} epochs")
print(f"Stage 2: {STAGE2_EPOCHS} epochs")
print(f"Batch size: {BATCH_SIZE}")
print(f"Output: {OUTPUT_DIR}")

## 4. Train SALT on MVTec AD

In [ ]:
# Run the full training pipeline
# This trains Stage 1 (MAE) + Stage 2 (JEPA), exports ONNX, generates all web data

!python scripts/generate_mvtec_data.py \
    --output-dir {OUTPUT_DIR} \
    --data-root {DATA_ROOT} \
    --stage1-epochs {STAGE1_EPOCHS} \
    --stage2-epochs {STAGE2_EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --device cuda

## 5. Verify Outputs

In [ ]:
import os
import json

output_dir = OUTPUT_DIR

# Check all output files
files = {
    "ONNX Model": f"{output_dir}/models/salt-inspector.onnx",
    "Training Metrics": f"{output_dir}/data/training-metrics.json",
    "Reference Embeddings": f"{output_dir}/data/reference-embeddings.json",
    "Sample Images": f"{output_dir}/data/mvtec-samples.json",
}

print("Output files:")
for name, path in files.items():
    if os.path.exists(path):
        size = os.path.getsize(path)
        if size > 1024 * 1024:
            print(f"  ✅ {name}: {size / (1024*1024):.1f} MB")
        else:
            print(f"  ✅ {name}: {size / 1024:.0f} KB")
    else:
        print(f"  ❌ {name}: NOT FOUND")

# Show training summary
with open(f"{output_dir}/data/training-metrics.json") as f:
    metrics = json.load(f)

print(f"\nTraining Summary:")
print(f"  Total time: {metrics.get('total_time_seconds', 0)}s")
print(f"  Stage 1 final loss: {metrics['stage1'][-1]['loss']}")
print(f"  Stage 2 final loss: {metrics['stage2'][-1]['loss']}")
print(f"  Model config: {metrics['config']}")

# Check reference embeddings
with open(f"{output_dir}/data/reference-embeddings.json") as f:
    refs = json.load(f)

print(f"\nReference Embeddings:")
for cat, data in refs['references'].items():
    print(f"  {cat}: {data['num_samples']} reference images")

## 6. Quick Anomaly Detection Test

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/content/salt-inspect')

from src.ijepa.models.encoder import build_encoder

# Load the trained student model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = {
    'img_size': 224, 'patch_size': 16, 'embed_dim': 192,
    'depth': 12, 'num_heads': 3, 'mlp_ratio': 4.0,
}
student = build_encoder(config).to(device)
student.load_state_dict(torch.load('/tmp/salt_inspector.pt', map_location=device))
student.eval()

# Load reference embeddings
import json
with open(f'{OUTPUT_DIR}/data/reference-embeddings.json') as f:
    ref_data = json.load(f)

# Test with a defective image
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Pick a defective test image
test_category = 'metal_nut'
test_dir = f'{DATA_ROOT}/mvtec_anomaly_detection/{test_category}/test'
defect_dirs = [d for d in os.listdir(test_dir) if d != 'good']

if defect_dirs:
    defect_dir = os.path.join(test_dir, defect_dirs[0])
    test_img_path = sorted(os.listdir(defect_dir))[0]
    test_img = Image.open(os.path.join(defect_dir, test_img_path)).convert('RGB')
    
    # Run inference
    img_tensor = test_transform(test_img).unsqueeze(0).to(device)
    with torch.no_grad():
        patch_embs = student(img_tensor)  # (1, 196, 192)
        patch_embs = F.normalize(patch_embs, dim=2)
    
    # Compare against normal references
    ref_patches = np.array(ref_data['references'][test_category]['patch_embeddings'])
    ref_patches = torch.tensor(ref_patches, device=device).float()
    
    # Per-patch cosine similarity
    similarity = (patch_embs[0] * ref_patches).sum(dim=1)  # (196,)
    anomaly_scores = 1.0 - similarity.cpu().numpy()  # Higher = more anomalous
    
    # Reshape to 14×14 grid
    heatmap = anomaly_scores.reshape(14, 14)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(test_img)
    axes[0].set_title(f'{test_category} ({defect_dirs[0]})')
    axes[0].axis('off')
    
    axes[1].imshow(heatmap, cmap='RdYlGn_r', vmin=0)
    axes[1].set_title('Anomaly Heatmap (14×14 patches)')
    axes[1].axis('off')
    
    # Overlay
    axes[2].imshow(test_img.resize((224, 224)))
    heatmap_resized = np.array(Image.fromarray(
        (heatmap * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR))
    axes[2].imshow(heatmap_resized, cmap='jet', alpha=0.4, vmin=0)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/anomaly_test.png', dpi=100)
    plt.show()
    
    print(f"\nAnomaly score: {anomaly_scores.mean():.4f} (higher = more anomalous)")
    print(f"Max patch score: {anomaly_scores.max():.4f}")
else:
    print(f"No defect directories found for {test_category}")

## 7. Download Outputs

Download the generated files to your local machine, then place them in the `site/` directory of your project.

In [ ]:
# Create a zip of all outputs for easy download
import shutil

shutil.make_archive('/content/salt-inspector-outputs', 'zip', OUTPUT_DIR)
print(f"Created /content/salt-inspector-outputs.zip")

zip_size = os.path.getsize('/content/salt-inspector-outputs.zip')
print(f"Size: {zip_size / (1024*1024):.1f} MB")

# Download in Colab
try:
    from google.colab import files
    files.download('/content/salt-inspector-outputs.zip')
except ImportError:
    print("Not in Colab — copy the zip manually")

In [ ]:
# Alternative: download individual files
try:
    from google.colab import files
    files.download(f'{OUTPUT_DIR}/models/salt-inspector.onnx')
    files.download(f'{OUTPUT_DIR}/data/training-metrics.json')
    files.download(f'{OUTPUT_DIR}/data/reference-embeddings.json')
    files.download(f'{OUTPUT_DIR}/data/mvtec-samples.json')
except ImportError:
    print("Not in Colab")